# Pivotowanie w pandas i polars — pivot_table, pivot, melt, stack/unstack, crosstab, pułapki

**Problem:** pivotowanie (przekształcanie danych z formatu długiego na szeroki i z powrotem) ma w obu bibliotekach kilka nakładających się na siebie narzędzi (`pivot`, `pivot_table`, `crosstab`, `stack`/`unstack`), które łatwo pomylić — zwłaszcza że różnią się tym, czy agregują duplikaty, czy się na nich wywalają.

**Porównanie:**
- `pd.pivot()` / `pl.DataFrame.pivot()` bez `aggregate_function` — czysty reshape, **wymaga unikalnych kombinacji** indeksu i kolumny; przy duplikatach rzuca błąd.
- `pd.pivot_table()` / `pl.DataFrame.pivot(..., aggregate_function=...)` — reshape **z agregacją**, radzi sobie z duplikatami.
- `melt()` / `unpivot()` — operacja odwrotna: z formatu szerokiego z powrotem do długiego.

**Kiedy stosować:** `pivot`/`pivot()` bez agregacji, gdy wiesz na pewno, że kombinacje są unikalne (i chcesz, żeby duplikat był zgłoszony jako błąd, nie po cichu zsumowany); `pivot_table`/`pivot(aggregate_function=...)`, gdy dane mają wiele wierszy na tę samą komórkę wynikowego pivota.

## Setup

Region `Central` ma dane tylko za styczeń — celowo, żeby zademonstrować obsługę brakujących kombinacji w dalszych sekcjach.

In [26]:
import pandas as pd
import polars as pl

data = {
    "region": ["North", "North", "North", "South", "South", "South",
               "East", "East", "West", "West", "Central"],
    "month": ["Jan", "Jan", "Feb", "Jan", "Jan", "Feb",
              "Jan", "Feb", "Jan", "Feb", "Jan"],
    "category": ["A", "B", "A", "A", "B", "A", "A", "B", "B", "A", "A"],
    "sales": [1200, 300, 1100, 900, 1500, 1000, 700, 950, 1300, 800, 600],
    "units": [12, 3, 11, 9, 15, 10, 7, 10, 13, 8, 6],
}

df = pd.DataFrame(data)
df_pl = pl.DataFrame(data)

df

,region,month,category,sales,units
0,North,Jan,A,1200,12
1,North,Jan,B,300,3
2,North,Feb,A,1100,11
3,South,Jan,A,900,9
4,South,Jan,B,1500,15
5,South,Feb,A,1000,10
6,East,Jan,A,700,7
7,East,Feb,B,950,10
8,West,Jan,B,1300,13
9,West,Feb,A,800,8


## Sekcja 1 — Podstawy: `pivot_table()` (pandas) / `pivot()` (polars)

Dane mają dla `(region, month)` po kilka wierszy (różne `category`), więc potrzebna jest agregacja — `aggfunc`/`aggregate_function` mówi, jak połączyć wiele wartości w jedną komórkę.

In [27]:
# pandas
pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum")

month,Feb,Jan
region,,
Central,NaN,600.0
East,950.0,700.0
North,1100.0,1500.0
South,1000.0,2400.0
West,800.0,1300.0


In [28]:
# polars
df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum")

region,Jan,Feb
str,i64,i64
"""North""",1500,1100
"""South""",2400,1000
"""East""",700,950
"""West""",1300,800
"""Central""",600,0


## Sekcja 2 — Braki po pivotowaniu: `fill_value` (pandas) vs `null`/`0` (polars)

`Central` nie ma danych za luty — po pivotowaniu ta komórka jest pusta. **Sposób, w jaki każda biblioteka to "puste" reprezentuje, różni się w zależności od funkcji agregującej** — to realna pułapka, patrz Sekcja 8.

In [29]:
# pandas - domyślnie NaN, jawny fill_value zamienia na wybraną wartość
print(pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum"))
print()
print(pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum", fill_value=0))

month       Feb     Jan
region                 
Central     NaN   600.0
East      950.0   700.0
North    1100.0  1500.0
South    1000.0  2400.0
West      800.0  1300.0

month     Feb   Jan
region             
Central     0   600
East      950   700
North    1100  1500
South    1000  2400
West      800  1300


In [30]:
# polars - .fill_null() po pivotowaniu, jeśli w ogóle potrzebne (patrz pułapka w Sekcji 8)
df_pl.pivot(on="month", index="region", values="sales", aggregate_function="mean").fill_null(0)

region,Jan,Feb
str,f64,f64
"""North""",750.0,1100.0
"""South""",1200.0,1000.0
"""East""",700.0,950.0
"""West""",1300.0,800.0
"""Central""",600.0,0.0


## Sekcja 3 — Wiele kolumn wartości naraz

pandas tworzy w wyniku dwupoziomowy `MultiIndex` kolumn (`(sales, Jan)`, `(sales, Feb)`, `(units, Jan)`...) — trzeba go spłaszczyć ręcznie przed dalszym użyciem (np. eksportem do CSV). polars robi to automatycznie, generując płaskie nazwy typu `sales_Jan`.

In [31]:
# pandas - MultiIndex kolumn
pivoted = pd.pivot_table(
    df, index="region", columns="month", values=["sales", "units"], aggfunc="sum", fill_value=0
)
print(pivoted.columns.tolist())
pivoted

[('sales', 'Feb'), ('sales', 'Jan'), ('units', 'Feb'), ('units', 'Jan')]


sales       units    
month     Feb   Jan   Feb Jan
region                       
Central     0   600     0   6
East      950   700    10   7
North    1100  1500    11  15
South    1000  2400    10  24
West      800  1300     8  13

In [32]:
# Spłaszczenie MultiIndex kolumn do pojedynczego poziomu z czytelnymi nazwami
pivoted.columns = [f"{value}_{month}" for value, month in pivoted.columns]
pivoted

,sales_Feb,sales_Jan,units_Feb,units_Jan
region,,,,
Central,0,600,0,6
East,950,700,10,7
North,1100,1500,11,15
South,1000,2400,10,24
West,800,1300,8,13


In [33]:
# polars - płaskie nazwy kolumn od razu, bez dodatkowego kroku
df_pl.pivot(on="month", index="region", values=["sales", "units"], aggregate_function="sum")

region,sales_Jan,sales_Feb,units_Jan,units_Feb
str,i64,i64,i64,i64
"""North""",1500,1100,15,11
"""South""",2400,1000,24,10
"""East""",700,950,7,10
"""West""",1300,800,13,8
"""Central""",600,0,6,0


### Różna agregacja dla różnych kolumn wartości: `aggfunc` jako słownik

Czasem `sales` ma sens sumować, a `units` uśredniać (np. średnia wielkość zamówienia) — w jednym wywołaniu `pivot_table`, przez podanie `aggfunc` jako słownika `{kolumna: funkcja}`.

In [34]:
pd.pivot_table(
    df, index="region", columns="month",
    values=["sales", "units"],
    aggfunc={"sales": "sum", "units": "mean"},
    fill_value=0,
)

sales       units      
month     Feb   Jan   Feb   Jan
region                         
Central     0   600   0.0   6.0
East      950   700  10.0   7.0
North    1100  1500  11.0   7.5
South    1000  2400  10.0  12.0
West      800  1300   8.0  13.0

### Kilka agregacji dla TEJ SAMEJ kolumny wartości: `aggfunc` jako lista

Gdy potrzebujesz jednocześnie sumy, średniej i liczby transakcji z tej samej kolumny — lista funkcji zamiast pojedynczej nazwy, bez wielokrotnego wywoływania `pivot_table`.

In [35]:
pd.pivot_table(
    df, index="region", columns="month",
    values="sales", aggfunc=["sum", "mean", "count"], fill_value=0,
)

sum          mean         count    
month     Feb   Jan     Feb     Jan   Feb Jan
region                                       
Central     0   600     0.0   600.0     0   1
East      950   700   950.0   700.0     1   1
North    1100  1500  1100.0   750.0     1   2
South    1000  2400  1000.0  1200.0     1   2
West      800  1300   800.0  1300.0     1   1

### Wiele kolumn `index` naraz: zagnieżdżone wiersze

`index=["region", "category"]` tworzy hierarchiczny (`MultiIndex`) układ wierszy — dwa poziomy grupowania po lewej stronie tabeli, zamiast jednego. Naturalne rozszerzenie tego, co już widziałeś przy `columns=` — tu ta sama logika działa dla wierszy.

In [36]:
pd.pivot_table(
    df, index=["region", "category"], columns="month",
    values="sales", aggfunc="sum", fill_value=0,
)

month              Feb   Jan
region  category            
Central A            0   600
East    A            0   700
        B          950     0
North   A         1100  1200
        B            0   300
South   A         1000   900
        B            0  1500
West    A          800     0
        B            0  1300

### To samo w polars: wiele kolumn `index` naraz

polars przyjmuje listę w `index=` dokładnie tak samo — różnica jest tylko wizualna: zamiast hierarchicznego `MultiIndex` po lewej stronie, `region` i `category` zostają zwykłymi, płaskimi kolumnami (bo polars w ogóle nie ma pojęcia indeksu).

In [37]:
df_pl.pivot(on="month", index=["region", "category"], values="sales", aggregate_function="sum")

region,category,Jan,Feb
str,str,i64,i64
"""North""","""A""",1200,1100
"""North""","""B""",300,0
"""South""","""A""",900,1000
"""South""","""B""",1500,0
"""East""","""A""",700,0
"""East""","""B""",0,950
"""West""","""B""",1300,0
"""West""","""A""",0,800
"""Central""","""A""",600,0


## Sekcja 4 — Sumy brzegowe (`margins`)

pandas ma wbudowany parametr `margins=True`, który dokłada wiersz i kolumnę z sumą całkowitą. **polars nie ma odpowiednika** — trzeba dodać wiersz/kolumnę sumaryczną ręcznie.

In [38]:
# pandas
pd.pivot_table(
    df, index="region", columns="month", values="sales", aggfunc="sum",
    fill_value=0, margins=True, margins_name="Razem",
)

month,Feb,Jan,Razem
region,,,
Central,0,600,600
East,950,700,1650
North,1100,1500,2600
South,1000,2400,3400
West,800,1300,2100
Razem,3850,6500,10350


In [39]:
# polars - ręczne dodanie kolumny i wiersza sumy
result = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum").fill_null(0)

month_cols = [c for c in result.columns if c != "region"]
result = result.with_columns(pl.sum_horizontal(month_cols).alias("Razem"))

total_row = result.select([pl.lit("Razem").alias("region")] + [pl.col(c).sum() for c in month_cols + ["Razem"]])
pl.concat([result, total_row])

region,Jan,Feb,Razem
str,i64,i64,i64
"""North""",1500,1100,2600
"""South""",2400,1000,3400
"""East""",700,950,1650
"""West""",1300,800,2100
"""Central""",600,0,600
"""Razem""",6500,3850,10350


## Sekcja 5 — `pivot()` bez agregacji: różnica względem `pivot_table()`

`pivot()`/`pivot(aggregate_function=None)` **zakłada unikalne kombinacje** indeksu i kolumny — nic nie agreguje. To zaleta, jeśli chcesz, żeby duplikat był głośnym błędem, a nie cichą sumą (patrz też Sekcja 8).

In [40]:
# pandas - unikalna kombinacja (region, category) -> działa bez agregacji
df.pivot(index=["region", "category"], columns="month", values="sales")

month                Feb     Jan
region  category                
Central A            NaN   600.0
East    A            NaN   700.0
        B          950.0     NaN
North   A         1100.0  1200.0
        B            NaN   300.0
South   A         1000.0   900.0
        B            NaN  1500.0
West    A          800.0     NaN
        B            NaN  1300.0

## Sekcja 6 — `melt()` / `unpivot()`: odwrotność pivota

Format szeroki z powrotem do długiego — przydatne np. przed wykresem w seaborn/plotly, który oczekuje jednej kolumny z wartością i jednej z kategorią.

In [41]:
wide = pd.pivot_table(df, index="region", columns="month", values="sales", aggfunc="sum", fill_value=0).reset_index()
print(wide)
print()

# pandas
wide.melt(id_vars="region", var_name="month", value_name="sales")

month   region   Feb   Jan
0      Central     0   600
1         East   950   700
2        North  1100  1500
3        South  1000  2400
4         West   800  1300



,region,month,sales
0,Central,Feb,0
1,East,Feb,950
2,North,Feb,1100
3,South,Feb,1000
4,West,Feb,800
5,Central,Jan,600
6,East,Jan,700
7,North,Jan,1500
8,South,Jan,2400
9,West,Jan,1300


In [42]:
# polars - nazywa się unpivot (dawniej też melt, przestarzałe)
wide_pl = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum").fill_null(0)
wide_pl.unpivot(index="region", variable_name="month", value_name="sales")

region,month,sales
str,str,i64
"""North""","""Jan""",1500
"""South""","""Jan""",2400
"""East""","""Jan""",700
"""West""","""Jan""",1300
"""Central""","""Jan""",600
"""North""","""Feb""",1100
"""South""","""Feb""",1000
"""East""","""Feb""",950
"""West""","""Feb""",800


## Sekcja 7 — `stack()` / `unstack()` (pandas, dane z `MultiIndex`)

Gdy dane już mają hierarchiczny indeks (np. wynik `groupby()` po dwóch kolumnach), `unstack()` przenosi jeden poziom indeksu do kolumn — de facto alternatywa dla `pivot_table` na danych już zgrupowanych. `stack()` robi odwrotność.

In [43]:
grouped = df.groupby(["region", "month"])["sales"].sum()
print(grouped)
print()

unstacked = grouped.unstack()
print(unstacked)
print()

unstacked.stack()

region   month
Central  Jan       600
East     Feb       950
         Jan       700
North    Feb      1100
         Jan      1500
South    Feb      1000
         Jan      2400
West     Feb       800
         Jan      1300
Name: sales, dtype: int64

month       Feb     Jan
region                 
Central     NaN   600.0
East      950.0   700.0
North    1100.0  1500.0
South    1000.0  2400.0
West      800.0  1300.0



region   month
Central  Jan       600.0
East     Feb       950.0
         Jan       700.0
North    Feb      1100.0
         Jan      1500.0
South    Feb      1000.0
         Jan      2400.0
West     Feb       800.0
         Jan      1300.0
dtype: float64

## Sekcja 8 — `crosstab()` (pandas): skrót do tabeli częstości

Gdy interesuje Cię tylko **liczba wystąpień** kombinacji (nie suma jakiejś wartości), `crosstab()` jest krótsze niż `pivot_table(..., aggfunc="count")`. Przyjmuje też `values=`/`aggfunc=`, jeśli jednak potrzebna agregacja innej kolumny.

In [44]:
# Liczba transakcji w każdej kombinacji region x month
print(pd.crosstab(df["region"], df["month"]))
print()

# To samo co pivot_table z aggfunc='sum', ale krócej
pd.crosstab(df["region"], df["month"], values=df["sales"], aggfunc="sum")

month    Feb  Jan
region           
Central    0    1
East       1    1
North      1    2
South      1    2
West       1    1



month,Feb,Jan
region,,
Central,NaN,600.0
East,950.0,700.0
North,1100.0,1500.0
South,1000.0,2400.0
West,800.0,1300.0


### `crosstab` z `normalize`: proporcje zamiast liczby wystąpień

Zamiast surowych liczb, `normalize=` przelicza tabelę na proporcje — trzy różne "punkty odniesienia" do wyboru, w zależności od pytania, na które odpowiadasz:
- `normalize="index"` — proporcje W RAMACH każdego wiersza (sumują się do 1 w poziomie). Pytanie: "jaki % sprzedaży regionu N przypada na kategorię A?"
- `normalize="columns"` — proporcje W RAMACH każdej kolumny (sumują się do 1 w pionie). Pytanie: "jaki % całej kategorii A pochodzi z regionu N?"
- `normalize="all"` — proporcje z CAŁEJ tabeli naraz (wszystkie komórki sumują się do 1). Pytanie: "jaki % wszystkich transakcji to akurat region N + kategoria A?"

In [45]:
print("normalize='index' - proporcje w ramach WIERSZA:")
print(pd.crosstab(df["region"], df["category"], normalize="index").round(2))
print()
print("normalize='columns' - proporcje w ramach KOLUMNY:")
print(pd.crosstab(df["region"], df["category"], normalize="columns").round(2))
print()
print("normalize='all' - proporcje z CAŁEJ tabeli:")
print(pd.crosstab(df["region"], df["category"], normalize="all").round(2))

normalize='index' - proporcje w ramach WIERSZA:
category     A     B
region              
Central   1.00  0.00
East      0.50  0.50
North     0.67  0.33
South     0.67  0.33
West      0.50  0.50

normalize='columns' - proporcje w ramach KOLUMNY:
category     A     B
region              
Central   0.14  0.00
East      0.14  0.25
North     0.29  0.25
South     0.29  0.25
West      0.14  0.25

normalize='all' - proporcje z CAŁEJ tabeli:
category     A     B
region              
Central   0.09  0.00
East      0.09  0.09
North     0.18  0.09
South     0.18  0.09
West      0.09  0.09


### `crosstab` z `margins`: sumy brzegowe też dla tabeli częstości

Ten sam parametr co w `pivot_table` (Sekcja 4) — dokłada wiersz i kolumnę "Razem".

In [46]:
pd.crosstab(df["region"], df["category"], margins=True, margins_name="Razem")

category,A,B,Razem
region,,,
Central,1,0,1
East,1,1,2
North,2,1,3
South,2,1,3
West,1,1,2
Razem,7,4,11


### `unstack` z `fill_value`: to samo co `fill_value` w `pivot_table`, tylko na już zgrupowanych danych

Domyślnie brakujące kombinacje po `unstack()` to `NaN` — dokładnie tak samo jak przy `pivot_table` bez `fill_value` (Sekcja 2).

In [47]:
grouped = df.groupby(["region", "month"])["sales"].sum()

print("Bez fill_value:")
print(grouped.unstack())
print()
print("Z fill_value=0:")
print(grouped.unstack(fill_value=0))

Bez fill_value:
month       Feb     Jan
region                 
Central     NaN   600.0
East      950.0   700.0
North    1100.0  1500.0
South    1000.0  2400.0
West      800.0  1300.0

Z fill_value=0:
month     Feb   Jan
region             
Central     0   600
East      950   700
North    1100  1500
South    1000  2400
West      800  1300


### polars: odpowiednik `crosstab` — `pivot` z `aggregate_function="len"`

polars nie ma osobnej funkcji `crosstab` — ten sam efekt (tabela częstości) osiąga się przez zwykły `pivot`, licząc długość grupy zamiast sumy wartości.

In [48]:
df_pl.pivot(
    on="category", index="region", values="category", aggregate_function="len"
).fill_null(0)

region,A,B
str,u32,u32
"""North""",2,1
"""South""",2,1
"""East""",1,1
"""West""",1,1
"""Central""",1,0


## Sekcja 9 — Pułapki

### Pułapka 1 — `pivot()` bez agregacji rzuca błąd przy duplikatach (w obu bibliotekach)

To zachowanie jest właściwie zaletą — `pivot()` **nie zgadnie za Ciebie**, jak połączyć duplikaty, tylko głośno się zatrzyma. Jeśli oczekujesz błędu, a dostajesz wynik bez niego, to sygnał, że przypadkiem użyłeś `pivot_table()`/`aggregate_function` zamiast czystego `pivot()`.

In [49]:
# pandas: (region, month) ma duplikaty (różne category) -> błąd
try:
    df.pivot(index="region", columns="month", values="sales")
except ValueError as e:
    print(f"pandas pivot() - błąd: {e}")

print()

# polars: to samo bez aggregate_function
try:
    df_pl.pivot(on="month", index="region", values="sales")
except Exception as e:
    print(f"polars pivot() bez aggregate_function - błąd: {type(e).__name__}")

pandas pivot() - błąd: Index contains duplicate entries, cannot reshape

polars pivot() bez aggregate_function - błąd: ComputeError


### Pułapka 2 — polars: `sum` traktuje brak danych jako `0`, `mean` jako `null`

To rozbieżność, która łatwo umyka: przy `aggregate_function="sum"` brakująca kombinacja (`Central`/`Feb`) dostaje **`0`**, a nie `null` — bo suma pustego zbioru to matematycznie zero. Przy `aggregate_function="mean"` ta sama brakująca kombinacja to **`null`** — bo średniej z pustego zbioru nie da się policzyć. Efekt: `0` po `sum` wygląda identycznie jak prawdziwa, zarejestrowana sprzedaż zerowa — nie da się ich odróżnić bez sprawdzenia danych źródłowych.

In [50]:
sum_pivot = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="sum")
mean_pivot = df_pl.pivot(on="month", index="region", values="sales", aggregate_function="mean")

print("aggregate_function='sum' - Central/Feb:")
print(sum_pivot.filter(pl.col("region") == "Central"))

print("\naggregate_function='mean' - Central/Feb:")
print(mean_pivot.filter(pl.col("region") == "Central"))

aggregate_function='sum' - Central/Feb:
shape: (1, 3)
┌─────────┬─────┬─────┐
│ region  ┆ Jan ┆ Feb │
│ ---     ┆ --- ┆ --- │
│ str     ┆ i64 ┆ i64 │
╞═════════╪═════╪═════╡
│ Central ┆ 600 ┆ 0   │
└─────────┴─────┴─────┘

aggregate_function='mean' - Central/Feb:
shape: (1, 3)
┌─────────┬───────┬──────┐
│ region  ┆ Jan   ┆ Feb  │
│ ---     ┆ ---   ┆ ---  │
│ str     ┆ f64   ┆ f64  │
╞═════════╪═══════╪══════╡
│ Central ┆ 600.0 ┆ null │
└─────────┴───────┴──────┘


### Pułapka 3 — `fill_value`/`fill_null` nie odróżnia "braku danych" od "wartości NaN w źródle"

Jeśli w danych źródłowych `sales` bywa `NaN` (np. niekompletny wpis, nie brak kombinacji), po `fill_value=0` obie sytuacje — "nie było takiego wiersza" i "był wiersz, ale z brakującą wartością" — wyglądają identycznie w wyniku. Zanim zastosujesz `fill_value`, warto sprawdzić `df["sales"].isna().sum()` na danych źródłowych, żeby wiedzieć, czy to rozróżnienie ma znaczenie w Twoim przypadku.

## Podsumowanie

| Zadanie | pandas | polars |
|---|---|---|
| Pivot z agregacją (duplikaty OK) | `pd.pivot_table(df, index=, columns=, values=, aggfunc=)` | `df.pivot(on=, index=, values=, aggregate_function=)` |
| Pivot bez agregacji (duplikaty = błąd) | `df.pivot(index=, columns=, values=)` | `df.pivot(on=, index=, values=)` (bez `aggregate_function`) |
| Wypełnienie braków po pivotowaniu | `fill_value=0` w `pivot_table` | `.fill_null(0)` po `pivot` |
| Wiele kolumn wartości naraz | `values=["a","b"]` → wymaga spłaszczenia `MultiIndex` kolumn | `values=["a","b"]` → płaskie nazwy od razu |
| Różna agregacja dla różnych kolumn wartości | `aggfunc={"a":"sum","b":"mean"}` | osobne wywołania `pivot` lub `group_by().agg()` |
| Kilka agregacji dla tej samej kolumny wartości | `aggfunc=["sum","mean","count"]` | osobne wywołania `pivot` lub `group_by().agg()` |
| Wiele kolumn `index` naraz (zagnieżdżone wiersze) | `index=["a","b"]` → hierarchiczny `MultiIndex` | `index=["a","b"]` → zwykłe, płaskie kolumny |
| Sumy brzegowe (wiersz/kolumna "Razem") | `margins=True` | brak wbudowanego — dodaj ręcznie (`sum_horizontal` + wiersz sumy) |
| Format szeroki → długi | `df.melt(id_vars=, var_name=, value_name=)` | `df.unpivot(index=, variable_name=, value_name=)` |
| Reshape na już zgrupowanych danych z `MultiIndex` | `.unstack(fill_value=)` / `.stack()` | nie dotyczy (polars nie ma indeksu) |
| Tabela częstości (liczba wystąpień) | `pd.crosstab(a, b)` | `df.pivot(..., aggregate_function="len")` |
| Tabela częstości jako proporcje, nie liczby | `pd.crosstab(a, b, normalize="index"/"columns"/"all")` | brak wprost — podziel ręcznie przez sumę |
| Sumy brzegowe w tabeli częstości | `pd.crosstab(a, b, margins=True)` | brak wprost |

**Wniosek:** najbezpieczniejszy nawyk to domyślnie sięgać po `pivot()` (bez agregacji) tam, gdzie kombinacje *powinny* być unikalne — błąd przy duplikacie wychwyci problem jakości danych, zanim zostanie po cichu zsumowany przez `pivot_table()`. `pivot_table()`/`aggregate_function` zostaw świadomie dla przypadków, gdzie agregacja jest zamierzona.